## S7: EUR conversion + per-year money split

Permanent, rerunnable notebook (unlike the one-off `funding_curated_gap_backfill.ipynb`) - future
Grants Tracker imports will keep arriving without a EUR figure, so this is meant to be run again
whenever new gaps show up, not just once.

1. **EUR conversion for Grants-Tracker rows** - Grants Tracker amounts are in their original
   currency; convert to EUR using the historical rate on the grant's `'Project start date'`, via
   the Frankfurter API (free, ECB-backed, no key).
2. **Per-year money split** - the existing convention in `funding_curated`'s `2020`-`2035` columns
   is `'Gov & NP contribution (EUR)'` split evenly across every year in `'Years active'` (verified
   exactly against multiple existing rows). Fill this in for any row that now has all three inputs
   but blank year columns.

Both sections only ever fill a currently-blank cell (`COALESCE`-gated) - safe to rerun on the whole
table each time.

**Caveat found while building this**: from my own sandboxed dev environment, `api.frankfurter.app`
returned `403 Forbidden` on every request (general internet access otherwise worked fine) - looked
like an IP/bot-protection block rather than the API being down, since a couple of other
Cloudflare-fronted FX services returned the same 403. I couldn't confirm live from here whether the
API call actually succeeds. Treat your first run of section 1 as the real test - if it also 403s
for you, let me know and I'll swap in a different rate source.

In [ ]:
import time
import urllib.request
import urllib.error
import json as _json
from datetime import datetime, timedelta
from pathlib import Path

import duckdb
import pandas as pd

DB_PATH = 'funding.db'
OUTPUT_DIR = Path('data_output')
OUTPUT_DIR.mkdir(exist_ok=True)
# Captured once so this run's output filename is consistent, however long the run takes.
RUN_TIMESTAMP = datetime.today().strftime('%y%m%d_%H%M')

# Non-ISO currency labels seen in the raw data - map to the ISO 4217 code Frankfurter expects.
CURRENCY_ALIASES = {'RMB': 'CNY', 'NIS': 'ILS'}

FRANKFURTER_BASE = 'https://api.frankfurter.app'
MAX_LOOKBACK_DAYS = 7  # walk back this many days if the exact date has no published rate (weekends/holidays)
REQUEST_TIMEOUT = 5  # seconds per attempt - halves the worst-case cost of one unresponsive pair
MAX_CONSECUTIVE_CONNECTION_FAILURES = 5  # abort if the network path itself looks down

# Frankfurter's bot protection rejects Python's default urllib User-Agent outright (403) - confirmed
# via direct testing that this is NOT an IP/network block (it 403'd identically from two different
# networks), just a client-fingerprint check. A normal browser-like header set gets through fine.
REQUEST_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
}


def parse_amount(x):
    """funding_curated stores 'Total amount'/'Gov contribution' as text, sometimes with space
    thousand-separators (e.g. '11 303 944.75') - strip those before float() rather than trusting the dtype."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    s = str(x).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() == 'nan':
        return None
    try:
        return float(s)
    except ValueError:
        return None


class FrankfurterUnreachable(Exception):
    """Raised after too many consecutive connection-level failures (not a clean HTTP response) -
    signals the network path itself is down, so there's no point grinding through every remaining
    pair at up to MAX_LOOKBACK_DAYS x REQUEST_TIMEOUT seconds each."""


_consecutive_connection_failures = 0


def fetch_eur_rate(currency, date_str):
    """Historical CCY->EUR rate on date_str (YYYY-MM-DD), walking back up to MAX_LOOKBACK_DAYS
    if the exact date has no published rate. Returns None if nothing is found for this pair
    specifically - callers should treat that as "couldn't convert this one", not raise. Raises
    FrankfurterUnreachable separately if the network itself looks unreachable (see above)."""
    global _consecutive_connection_failures
    if currency == 'EUR':
        _consecutive_connection_failures = 0
        return 1.0
    currency = CURRENCY_ALIASES.get(currency, currency)
    date = datetime.strptime(date_str, '%Y-%m-%d')
    for i in range(MAX_LOOKBACK_DAYS + 1):
        d = (date - timedelta(days=i)).strftime('%Y-%m-%d')
        url = f'{FRANKFURTER_BASE}/{d}?from={currency}&to=EUR'
        req = urllib.request.Request(url, headers=REQUEST_HEADERS)
        try:
            with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
                data = _json.loads(resp.read())
            _consecutive_connection_failures = 0  # got a real response - network path works
            rate = data.get('rates', {}).get('EUR')
            if rate is not None:
                return rate
        except urllib.error.HTTPError:
            _consecutive_connection_failures = 0  # also a real response (e.g. 404) - network is fine
        except (urllib.error.URLError, TimeoutError, OSError):
            # a raw connection-level failure (timeout/DNS/refused) - no HTTP response was ever
            # received, unlike HTTPError above. Some Python/OpenSSL versions raise a bare
            # TimeoutError here instead of wrapping it in URLError, so catch both explicitly.
            _consecutive_connection_failures += 1
            if _consecutive_connection_failures >= MAX_CONSECUTIVE_CONNECTION_FAILURES:
                raise FrankfurterUnreachable(
                    f"{_consecutive_connection_failures} consecutive connection failures - "
                    f"Frankfurter appears unreachable from this network."
                )
        time.sleep(0.1)  # be polite to the free API
    return None

### 1. EUR conversion for Grants-Tracker rows

In [ ]:
con = duckdb.connect(DB_PATH)

# Universal scope - any row with a real amount + known currency but a blank EUR figure, not just
# 'airtable'. Dimensions rows that already get 'Total amount (EUR)' natively from the API are
# unaffected (nothing to fill); this only kicks in for rows still missing it regardless of source.
candidates = con.sql('''
    SELECT lrd_row_id, "Identification code", "Total amount", "Gov contribution", "Currency", "Project start date"
    FROM funding_curated
    WHERE "Currency" IS NOT NULL
    AND "Project start date" IS NOT NULL
    AND ("Total amount (EUR)" IS NULL OR "Gov & NP contribution (EUR)" IS NULL)
''').df()

candidates['total_parsed'] = candidates['Total amount'].apply(parse_amount)
candidates['gov_parsed'] = candidates['Gov contribution'].apply(parse_amount)
candidates['date_str'] = pd.to_datetime(candidates['Project start date'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
n_unparseable_date = candidates['date_str'].isna().sum()
if n_unparseable_date:
    print(f"{n_unparseable_date} candidate rows have an unparseable 'Project start date' - skipping those.")
candidates = candidates[candidates['date_str'].notna()]

needs_conversion = candidates[
    (candidates['total_parsed'].notna() & (candidates['total_parsed'] != 0)) |
    (candidates['gov_parsed'].notna() & (candidates['gov_parsed'] != 0))
].copy()
print(f"{len(needs_conversion)} rows have a real amount to convert (of {len(candidates)} candidates).")

# Cache (currency, date) -> rate so repeated pairs across rows only hit the API once each. Progress
# printed every 20 pairs so a live run shows visible movement instead of going silent for the whole
# loop; aborts early (see FrankfurterUnreachable in the cell above) if the network looks down rather
# than grinding through every remaining pair at up to MAX_LOOKBACK_DAYS x REQUEST_TIMEOUT seconds each.
unique_pairs = needs_conversion[['Currency', 'date_str']].drop_duplicates().values.tolist()
rate_cache = {}
failed_pairs = set()
aborted = False
for i, (currency, date_str) in enumerate(unique_pairs, 1):
    try:
        rate = fetch_eur_rate(currency, date_str)
    except FrankfurterUnreachable as e:
        print(f"ABORTING: {e}")
        print(f"Stopped after {i - 1}/{len(unique_pairs)} pairs looked up - rerun once network access is confirmed.")
        aborted = True
        break
    if rate is None:
        failed_pairs.add((currency, date_str))
    rate_cache[(currency, date_str)] = rate
    if i % 20 == 0 or i == len(unique_pairs):
        print(f"  ...looked up {i}/{len(unique_pairs)} pairs so far ({len(failed_pairs)} failed)")

if not aborted:
    print(f"Looked up {len(rate_cache)} unique (currency, date) pairs, {len(failed_pairs)} failed.")
    if failed_pairs:
        print(f"Failed pairs (no rate found within {MAX_LOOKBACK_DAYS} days): {sorted(failed_pairs)[:10]}")

needs_conversion['rate'] = needs_conversion.apply(
    lambda r: rate_cache.get((r['Currency'], r['date_str'])), axis=1,
)
needs_conversion['total_amount_eur'] = needs_conversion['total_parsed'] * needs_conversion['rate']
needs_conversion['gov_contribution_eur'] = needs_conversion['gov_parsed'] * needs_conversion['rate']

n_converted = needs_conversion['rate'].notna().sum()
print(f"{n_converted}/{len(needs_conversion)} rows successfully converted; "
      f"{len(needs_conversion) - n_converted} skipped (no rate available).")

In [ ]:
# Universal join key - Dimensions-promoted rows have no lrd_row_id (only 'Identification code'),
# while Grants-Tracker/last-report rows always have an lrd_row_id - same COALESCE pattern used for
# the per-year split below, now needed here too since scope is no longer airtable-only.
patch = needs_conversion[['lrd_row_id', 'Identification code', 'total_amount_eur', 'gov_contribution_eur']].dropna(
    how='all', subset=['total_amount_eur', 'gov_contribution_eur']
).copy()
patch['join_key'] = patch['lrd_row_id'].astype(str).where(patch['lrd_row_id'].notna(), patch['Identification code'])

con.register('eur_patch', patch[['join_key', 'total_amount_eur', 'gov_contribution_eur']])
con.sql('''
    UPDATE funding_curated
    SET
        "Total amount (EUR)" = COALESCE(funding_curated."Total amount (EUR)", eur_patch.total_amount_eur),
        "Gov & NP contribution (EUR)" = COALESCE(funding_curated."Gov & NP contribution (EUR)", eur_patch.gov_contribution_eur)
    FROM eur_patch
    WHERE COALESCE(funding_curated.lrd_row_id::VARCHAR, funding_curated."Identification code") = eur_patch.join_key
''')
con.unregister('eur_patch')

n_now = con.sql('''
    SELECT
        SUM(CASE WHEN "Total amount" IS NOT NULL THEN 1 ELSE 0 END) AS has_total,
        SUM(CASE WHEN "Total amount (EUR)" IS NOT NULL THEN 1 ELSE 0 END) AS has_eur,
        SUM(CASE WHEN "Gov contribution" IS NOT NULL THEN 1 ELSE 0 END) AS has_gov,
        SUM(CASE WHEN "Gov & NP contribution (EUR)" IS NOT NULL THEN 1 ELSE 0 END) AS has_gov_eur,
        COUNT(*) AS n
    FROM funding_curated
''').df()
print(n_now.to_string())

### 2. Per-year money split

In [ ]:
YEAR_COLS = list(range(2020, 2036))
blank_year_check = ' OR '.join(f'"{y}" IS NULL' for y in YEAR_COLS)

# Dimensions-promoted rows have no lrd_row_id (only 'Identification code'), while Grants-Tracker/
# last-report rows always have an lrd_row_id - COALESCE gives one join key that works for either
# origin, since this section (unlike section 1 above) intentionally covers both.
rows = con.sql(f'''
    SELECT
        COALESCE(lrd_row_id::VARCHAR, "Identification code") AS join_key,
        "Gov & NP contribution (EUR)", "duration (years)", "Years active"
    FROM funding_curated
    WHERE "Gov & NP contribution (EUR)" IS NOT NULL
    AND "duration (years)" IS NOT NULL AND "duration (years)" > 0
    AND "Years active" IS NOT NULL
    AND ({blank_year_check})
''').df()
print(f"{len(rows)} rows have everything needed for a per-year split but at least one blank year cell.")

n_dupe_keys = rows['join_key'].duplicated().sum()
if n_dupe_keys:
    print(f"WARNING: {n_dupe_keys} duplicate join keys (blank lrd_row_id AND blank/shared "
          f"Identification code) - those rows may get another row's split values. Investigate before trusting them.")

per_year_amount = rows['Gov & NP contribution (EUR)'] / rows['duration (years)']
years_lists = rows['Years active'].apply(lambda s: [int(y.strip()) for y in str(s).split(',') if y.strip()])

split_df = pd.DataFrame({'join_key': rows['join_key']})
for y in YEAR_COLS:
    split_df[str(y)] = [amt if y in years else None for amt, years in zip(per_year_amount, years_lists)]

con.register('split_patch', split_df)
set_clause = ', '.join(f'"{y}" = COALESCE(funding_curated."{y}", split_patch."{y}")' for y in YEAR_COLS)
con.sql(f'''
    UPDATE funding_curated
    SET {set_clause}
    FROM split_patch
    WHERE COALESCE(funding_curated.lrd_row_id::VARCHAR, funding_curated."Identification code") = split_patch.join_key
''')
con.unregister('split_patch')

print(f"Per-year split written for {len(split_df)} rows.")

# Excel copy of the fully-updated funding_curated dataset, so every S7 run produces a ready-to-use
# analysis file without needing a one-off script - RUN_TIMESTAMP keeps each run's export distinct.
export_path = OUTPUT_DIR / f"{RUN_TIMESTAMP}_funding_curated_FINAL.xlsx"
con.sql("SELECT * FROM funding_curated").df().to_excel(export_path, index=False)
print(f"Excel copy of 'funding_curated' saved to {export_path}")

con.close()